# Genome-GPT — Interactive Exploration

This notebook walks through the full offline pipeline so you can play with
the model without needing NCBI or an API key.

**Prerequisites**
```bash
pip install -r requirements-dev.txt jupyter
```

## 1 — Build a synthetic corpus and train a tiny model

This takes ~2 minutes on CPU. Skip to section 2 if you already have a checkpoint.

In [ ]:
import subprocess
import sys


def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout or result.stderr)


run(f"{sys.executable} data/make_synthetic.py")
run(f"{sys.executable} data/prepare.py --fasta data/synth.fasta")

In [ ]:
# Train a small model — takes ~1-2 min on CPU
CKPT = "checkpoints/explore.pt"
run(
    f"{sys.executable} train.py "
    "--max_iters 800 --n_layer 3 --n_embd 128 --block_size 128 "
    f"--batch_size 32 --device cpu --ckpt_path {CKPT} "
    "--eval_interval 200 --sample_interval 400"
)

## 2 — Load the model

In [ ]:
from inference import GenomeModel

CKPT = "checkpoints/explore.pt"  # change to checkpoints/ckpt.pt for a full run
gm = GenomeModel(CKPT)
print(f"Loaded checkpoint. block_size={gm.cfg.block_size}  n_embd={gm.cfg.n_embd}")

## 3 — Score sequences

`bits_per_bp` is the key number. Random DNA = **2.0**; lower = more natural under
the model. On synthetic data nothing should beat ~2.0 by much — real bacteria
are where the gap appears.

In [ ]:
seqs = {
    "pure_random": "ATCGATCGATCGATCG" * 4,
    "all_A": "A" * 64,
    "alternating": "ACGT" * 16,
    "gc_rich": "GCGCGCGCGCGCGCGC" * 4,
}

for name, seq in seqs.items():
    r = gm.score(seq)
    print(f"{name:<16}  bits/bp={r['bits_per_bp']:.4f}  perplexity={r['perplexity']:.2f}")

## 4 — Generate novel DNA

In [ ]:
for temp in (0.6, 0.9, 1.2):
    seq = gm.generate(prompt="ATG", n_bases=80, temperature=temp, top_k=4, seed=0)
    gc = (seq.count("G") + seq.count("C")) / max(1, len(seq) - seq.count("|"))
    print(f"temp={temp}  GC={gc:.2f}  {seq[:60]}...")

## 5 — Variant effect prediction

**LLR < 0** means the substitution makes the sequence less likely (more disruptive).
Keep the variant centered in the window — edge positions score unreliably.

In [ ]:
ref = "ACGTACGTACGT" * 8  # 96-bp reference
pos = 48  # center

for alt in "ACGT":
    if alt == ref[pos]:
        continue
    r = gm.variant_effect(ref, pos=pos, alt_base=alt)
    print(f"{ref[pos]}->{alt}  LLR={r['llr']:+.4f}  ({r['interpretation']})")

## 6 — Embeddings and similarity

Cosine similarity between mean-pooled hidden states. Similar sequences should
cluster together even on a small synthetic model.

In [ ]:
import numpy as np


def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


seqs = {
    "gc_rich_1": "GCGCGCGCGCGCGCGC" * 4,
    "gc_rich_2": "CGCGCGCGCGCGCGCG" * 4,
    "at_rich": "ATATATATATATATA" * 4,
    "random": "ATCGATCGATCGATCG" * 4,
}

embeds = {name: gm.embed(seq) for name, seq in seqs.items()}
names = list(embeds)

print("Cosine similarity matrix")
print(f"{'':16}" + "".join(f"{n:>12}" for n in names))
for a in names:
    row = "".join(f"{cosine(embeds[a], embeds[b]):>12.4f}" for b in names)
    print(f"{a:<16}{row}")

## 7 — Markov baseline comparison

Reproduce the table from `evaluate.py` inline.

In [ ]:
from config import ITOS, Config
from evaluate import markov_bits

cfg = Config()
train_ids = np.fromfile(cfg.train_bin, dtype=np.uint8)
val_ids = np.fromfile(cfg.val_bin, dtype=np.uint8)[:50_000]
val_str = "".join(ITOS[int(i)] for i in val_ids)

neural = gm.score(val_str)["bits_per_bp"]

print(f"\n  {'model':<22}{'bits/token':>12}")
print(f"  {'-' * 34}")
print(f"  {'random (4 bases)':<22}{2.000:>12.4f}")
for k in (0, 2, 4, 6):
    b = markov_bits(train_ids, val_ids, k)
    print(f"  {'markov k=' + str(k):<22}{b:>12.4f}")
print(f"  {'neural GPT':<22}{neural:>12.4f}   <-- goal: beat best k-gram")